# putEMG — Feature-Based Cross-Subject Models (LOSO)

Mirrors `experiments/cross_subject/deep_learning/model.ipynb` but operates on
**hand-crafted features** (libemg, 50 feature groups × 24 channels = 3192-dim per window).
Each rep is converted to a `(26, 3192)` temporal sequence and classified by a sequence model
using **Leave-One-Subject-Out (LOSO)** cross-validation. Set `MODEL_TYPE` to switch models.

| Model | Description |
|-------|-------------|
| **FeatureLSTM** | Bidirectional LSTM over the 26-step feature sequence |
| **FeatureGRU** | Bidirectional GRU — lighter alternative to LSTM |
| **FeatureTransformer** | Self-attention encoder with sinusoidal positional encoding |

**LOSO split (per fold):**
- **Test** — 1 held-out subject (never seen during training)
- **Val** — 10% of the remaining 43 subjects, stratified by class (early stopping only)
- **Train** — remaining 90% of those 43 subjects

Weights are saved per model under `weights/<MODEL_TYPE>/`; per-fold logs under `results/results_log_<MODEL_TYPE>.txt`.
Completed folds are skipped automatically — safe to stop and resume at any time.

> **Prerequisites**: Run `data_preprocessing/driver.ipynb` first to generate per-subject `.mat` files.
> Features are extracted automatically by this notebook on first run.

In [ ]:
import os
import sys
import numpy as np
import datetime
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# -- Shared utilities (src/) --
sys.path.append(os.path.abspath('../../../'))

import src.feature_based_models as fbm
from src.emg_loader import (
    load_all_subjects,
    load_feature_subjects,
    make_loso_train_val_test,
)
from src.feature_extraction import batch_extract

In [ ]:
# ── Training & evaluation utilities ────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(model, train_loader, criterion, optimizer, device):
    """One full epoch over train_loader. Returns mean batch loss."""
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, loader, device):
    """Returns top-1 accuracy over loader. No gradient computation."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return correct / total


def evaluateFinal(model, test_loader, device, plot=True):
    """Full evaluation with optional confusion matrix — use for post-hoc analysis."""
    model.eval()
    correct, total = 0, 0
    actual    = torch.tensor([], dtype=torch.int64)
    predicted = torch.tensor([], dtype=torch.int64)
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct   += (pred == y).sum().item()
            total     += y.size(0)
            actual    = torch.cat((actual,    y.cpu()),    dim=0)
            predicted = torch.cat((predicted, pred.cpu()), dim=0)

    acc = correct / total
    if plot:
        print(np.unique(predicted.numpy(), return_counts=True))
        cm = confusion_matrix(actual.numpy(), predicted.numpy())
        ConfusionMatrixDisplay(cm).plot()
    print(f'Test accuracy: {acc * 100:.2f}%')
    return acc

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
_notebook_dir = os.path.abspath(os.getcwd())
print(_notebook_dir)

In [ ]:
# -- Paths --
DATA_DIR    = '/Volumes/KRIS/data/UG_per_subject'         # raw preprocessed .mat files
FEATURE_DIR = '/Volumes/KRIS/data/features_sequence'      # shared feature cache (.npz files)
MODE        = 'sequence'

MODEL_TYPE  = 'FeatureLSTM'   # change to: 'FeatureGRU' or 'FeatureTransformer'
WEIGHTS_DIR = os.path.join(_notebook_dir, 'weights', MODEL_TYPE)
RESULTS_DIR = os.path.join(_notebook_dir, 'results')
LOG_PATH    = os.path.join(RESULTS_DIR, f'results_log_{MODEL_TYPE}.txt')

# -- Data split --
BATCH_SIZE = 16
VAL_FRAC   = 0.10     # fraction of training pool held out for early stopping (stratified)

# -- Training --
MAX_EPOCHS = 20
PATIENCE   = 5
MIN_DELTA  = 0.002
LR         = 1e-3
DROPOUT    = 0.3

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)
print(f'Weights → {WEIGHTS_DIR}')
print(f'Log     → {LOG_PATH}')

---
## Feature Extraction

If `FEATURE_DIR` is empty or incomplete, extract features from the raw `.mat` files and cache them as `.npz`.
Subjects with existing files are skipped by `batch_extract` — safe to re-run.

In [ ]:
expected = len([f for f in os.listdir(DATA_DIR) if f.endswith('.mat')])
present  = len([f for f in os.listdir(FEATURE_DIR) if f.endswith(f'_{MODE}.npz')])
print(f'Raw subjects     : {expected}')
print(f'Cached features  : {present}  ({FEATURE_DIR})')

if present < expected:
    print(f'\nExtracting features for {expected - present} subject(s)\u2026')
    raw_subjects = load_all_subjects(DATA_DIR)
    batch_extract(raw_subjects, output_dir=FEATURE_DIR, mode=MODE)
else:
    print('All feature files present — skipping extraction.')

subjects = load_feature_subjects(FEATURE_DIR, mode=MODE)

---
## LOSO Training Loop

Trains the selected model (`MODEL_TYPE`) on every subject as a test fold:

- **Test** — 1 held-out subject, never seen during training
- **Train / Val** — all other 43 subjects, split 90/10 (stratified by class, seeded) within the pool

Checkpointing:
- Weights saved to `weights/<MODEL_TYPE>/<MODEL_TYPE>_<subject_id>.pt` after each fold
- If a checkpoint already exists the fold is skipped — safe to stop and resume at any time
- `results_log_<MODEL_TYPE>.txt` is rebuilt from all checkpoints after every fold

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────

MODEL_MAP = {
    'FeatureLSTM':        fbm.FeatureLSTM,
    'FeatureGRU':         fbm.FeatureGRU,
    'FeatureTransformer': fbm.FeatureTransformer,
}

def subject_id(filename):
    """'features_subject_03_sequence.npz' → '03'"""
    return filename.split('_')[2]


def weight_path(subject_name):
    return os.path.join(WEIGHTS_DIR, f'{MODEL_TYPE}_{subject_id(subject_name)}.pt')


def update_log():
    """Rebuild results_log_<MODEL_TYPE>.txt from all valid LOSO checkpoint files in WEIGHTS_DIR."""
    checkpoints = []
    for fname in sorted(os.listdir(WEIGHTS_DIR)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
        if 'subject' not in ckpt:
            continue
        checkpoints.append(ckpt)

    if not checkpoints:
        return

    accs    = [c['test_acc'] * 100 for c in checkpoints]
    n_done  = len(checkpoints)
    n_total = len(subjects)

    lines = [
        f'putEMG — {MODEL_TYPE} LOSO Results (feature-based)',
        '=' * 68,
        f"{'Subject':<32} {'Test Acc':>9}  {'Val Acc':>9}  {'Epoch':>6}  {'Date'}",
        '-' * 68,
    ]
    for c in checkpoints:
        lines.append(
            f"{c['subject']:<32} {c['test_acc']*100:>8.2f}%  "
            f"{c['val_acc']*100:>8.2f}%  {c['best_epoch']:>6}  {c['date']}"
        )
    lines += [
        '=' * 68,
        f"Mean: {np.mean(accs):.2f}%  \u00b1  {np.std(accs):.2f}%  "
        f"({n_done} / {n_total} folds complete)",
    ]

    with open(LOG_PATH, 'w') as f:
        f.write('\n'.join(lines) + '\n')

    print(f'Log updated \u2192 {LOG_PATH}  ({n_done}/{n_total} folds)')

In [ ]:
# ── LOSO outer loop: iterate over every subject as the held-out test fold ────
# Each iteration trains a fresh model on all OTHER subjects and tests on this one.
# Checkpointing: if a .pt file already exists for a subject, skip that fold —
# this makes it safe to interrupt and resume without re-training completed folds.
for test_idx, (test_name, _, _) in enumerate(subjects):
    wpath = weight_path(test_name)

    # Resume support: skip folds that already have a saved checkpoint
    if os.path.exists(wpath):
        print(f'[SKIP] {test_name}  — checkpoint found')
        continue

    print(f"\n{'='*60}")
    print(f'  Fold {test_idx+1}/{len(subjects)}  —  test: {test_name}')
    print(f"{'='*60}")

    # Build dataloaders for this fold:
    #   - test_loader  : only the held-out subject (never used during training)
    #   - train_loader : 90% of the remaining 43 subjects (stratified split)
    #   - val_loader   : the other 10%, used only for early stopping
    train_loader, val_loader, test_loader = make_loso_train_val_test(
        subjects, test_idx, val_frac=VAL_FRAC, batch_size=BATCH_SIZE
    )

    # Fresh model + optimizer for every fold — no weight leakage between subjects
    model     = MODEL_MAP[MODEL_TYPE](dropout_rate=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    # Halve LR when val accuracy plateaus; protects against overshooting the optimum
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    # Track the best validation checkpoint for early stopping
    best_val_acc = float('-inf')
    best_state   = None   # deep-copy of model weights at the best val epoch
    best_epoch   = 0
    bad_epochs   = 0      # consecutive epochs without a MIN_DELTA improvement

    # ── Inner loop: train for up to MAX_EPOCHS, stop early if val stagnates ──
    for epoch in range(MAX_EPOCHS):
        tr_loss = train(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        curr_lr = optimizer.param_groups[0]['lr']

        # Snapshot the weights whenever this epoch beats the previous best val accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch   = epoch + 1

        scheduler.step(val_acc)
        bad_epochs = 0 if val_acc >= (best_val_acc - MIN_DELTA) else bad_epochs + 1

        print(f'  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  val={val_acc*100:.2f}%  '
              f'best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}')

        if bad_epochs >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    # Restore the best-seen weights before evaluating on the held-out test subject
    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader, device)
    print(f'\n  Test accuracy: {test_acc*100:.2f}%')

    # Save checkpoint with metadata so update_log() can rebuild the results file
    torch.save({
        'subject':    test_name,
        'test_acc':   test_acc,
        'val_acc':    best_val_acc,
        'best_epoch': best_epoch,
        'dropout':    DROPOUT,
        'state_dict': best_state,
        'date':       datetime.date.today().isoformat(),
    }, wpath)

    update_log()

print(f"\n{'='*60}")
print('  All folds complete.')
print(f"{'='*60}")
update_log()

---
## Results

In [ ]:
# ── Print log ─────────────────────────────────────────────────────────
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        print(f.read())
else:
    print('No results yet — run the training loop first.')

# ── Bar chart ─────────────────────────────────────────────────────────
checkpoints = []
for fname in sorted(os.listdir(WEIGHTS_DIR)):
    if not fname.endswith('.pt'):
        continue
    ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
    if 'subject' not in ckpt:
        continue
    checkpoints.append(ckpt)

if checkpoints:
    sids  = [subject_id(c['subject']) for c in checkpoints]
    accs  = [c['test_acc'] * 100 for c in checkpoints]
    mean  = np.mean(accs)

    fig, ax = plt.subplots(figsize=(max(10, len(sids) * 0.45), 4))
    ax.bar(sids, accs, color='steelblue')
    ax.axhline(mean, color='tomato', linestyle='--', linewidth=1.5, label=f'Mean {mean:.1f}%')
    ax.set_ylim(0, 105)
    ax.set_xlabel('Test Subject')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{MODEL_TYPE} LOSO — Per-Subject Accuracy  ({len(checkpoints)}/{len(subjects)} folds)')
    ax.legend()
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()